# House Prices - Advanced Regression Techniques
## Final Exam Project, Machine Learning

**Author**: Todor Yankov  
**Date**: May 2026  
**Goal**: Predict house sale prices (Ames, Iowa) using regression, classification, clustering, dimensionality reduction, and MLflow.

This notebook performs initial data exploration (EDA) and establishes a baseline model.

## 1. Load Libraries and Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

# Load data (adjust path if needed)
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

print("Train shape:", train.shape)
print("Test shape:", test.shape)

## 2. Initial Overview

In [ ]:
train.head()

In [ ]:
train.info()

In [ ]:
train.describe()

## 3. Missing Values

In [ ]:
missing_train = train.isnull().sum()
missing_train = missing_train[missing_train > 0].sort_values(ascending=False)
print("Missing values in train:\n", missing_train)

missing_test = test.isnull().sum()
missing_test = missing_test[missing_test > 0].sort_values(ascending=False)
print("Missing values in test:\n", missing_test)

## 4. Target Variable Distribution

In [ ]:
sns.histplot(train['SalePrice'], bins=50, kde=True)
plt.title('Distribution of SalePrice')
plt.xlabel('SalePrice')
plt.show()

print("Skewness:", train['SalePrice'].skew())
print("Kurtosis:", train['SalePrice'].kurtosis())

## 5. Summary for First Submission

The baseline model (Random Forest) used only 11 features with high correlation to `SalePrice`.  
This notebook will be extended with:
- Detailed visualizations (correlation matrix, feature importance)
- Cross-validation (RMSLE)
- Comparison of multiple models (XGBoost, LightGBM, Stacking)
- Final results and discussion

## 6. Correlation Analysis & Feature Importance

This section analyzes the correlation between numerical features and the target variable `SalePrice`, and compares it with the feature importance from the Random Forest model.

In [ ]:
# Compute correlation of all numerical columns with SalePrice
numeric_cols = train.select_dtypes(include=[np.number]).columns
corr_with_price = train[numeric_cols].corr()['SalePrice'].sort_values(ascending=False)

# Select top 11 features (excluding SalePrice itself)
top_features = corr_with_price[1:12]

# Plot the correlations
plt.figure(figsize=(10,6))
top_features.plot(kind='bar', color='steelblue')
plt.title('Top 11 Features Correlated with SalePrice', fontsize=14)
plt.ylabel('Pearson Correlation')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("Selected 11 features and their correlations:")
print(top_features)

## 7. Random Forest Feature Importances

*This section displays the feature importance scores extracted from the Random Forest model, confirming which variables most influence the predictions.*

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Define the 11 features
features = [
    'OverallQual', 'GrLivArea', 'GarageCars', 'GarageArea', 'TotalBsmtSF',
    '1stFlrSF', 'YearBuilt', 'YearRemodAdd', 'LotArea', 'FullBath', 'BedroomAbvGr'
]

# Prepare data
X_train_rf = train[features].fillna(0)
y_train_rf = np.log1p(train['SalePrice'])

# Train Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train_rf, y_train_rf)

# Get feature importances
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

# Plot
plt.figure(figsize=(10,6))
plt.bar(range(len(features)), importances[indices], align='center', color='darkorange')
plt.xticks(range(len(features)), [features[i] for i in indices], rotation=45, ha='right')
plt.title('Random Forest Feature Importances', fontsize=14)
plt.ylabel('Importance Score')
plt.tight_layout()
plt.show()

# Print importance values
print("Feature importances (Random Forest):")
for i in indices:
    print(f"{features[i]:15s} : {importances[i]:.4f}")

## 8. Cross-Validation of Baseline Model (RMSLE)

*Evaluating the stability of the Random Forest model using 5-fold cross-validation.*

In [ ]:
from sklearn.model_selection import cross_val_score

# Features and target
X = train[features].fillna(0)
y = np.log1p(train['SalePrice'])

# Model
rf_cv = RandomForestRegressor(n_estimators=100, random_state=42)

# Cross-validation (negative RMSE, we convert to positive)
scores = cross_val_score(rf_cv, X, y, cv=5, scoring='neg_root_mean_squared_error')
rmse_scores = -scores

print(f"RMSLE (5-fold CV): {rmse_scores.mean():.4f} +/- {rmse_scores.std()*2:.4f}")
print(f"Individual fold scores: {rmse_scores}")

## 9. Baseline Kaggle Result

The first Kaggle submission (Random Forest with 11 features) achieved **RMSLE = 0.15793** on the public leaderboard.  
This serves as a baseline for comparison with more advanced models (XGBoost, LightGBM, Stacking).